# Statistical analysis

In [1]:
# Dependencies
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

from scipy.optimize import curve_fit
from scipy.stats import spearmanr, wilcoxon

In [2]:
# Project directories

PROJECT_DIR = Path("..")

AGGREGATED_DIR = PROJECT_DIR / "data" / "aggregated"


# Input files

FULL_SWEEP = AGGREGATED_DIR / "full_sweep_means.csv"
MODEL_VARIANTS = AGGREGATED_DIR / "model_variants_means.csv"

## 1. FULL SWEEP

In [3]:
df = pd.read_csv(FULL_SWEEP)

### Multiple regression: JS divergence

In [4]:
# Rename variables for analysis

full_sweep = df.rename(
    columns={
        "mean_risk_perception_final_mean": "risk_perception",
        "mean_js_final_mean": "js_divergence",
    }
)

model = smf.ols(
    "js_divergence ~ social_memory_size + p_outgroup + risk_perception",
    data=full_sweep,
).fit()

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:          js_divergence   R-squared:                       0.812
Model:                            OLS   Adj. R-squared:                  0.794
Method:                 Least Squares   F-statistic:                     46.02
Date:                Thu, 01 Oct 2026   Prob (F-statistic):           1.04e-11
Time:                        10:19:58   Log-Likelihood:                 50.824
No. Observations:                  36   AIC:                            -93.65
Df Residuals:                      32   BIC:                            -87.31
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept             -0.0148      0

### AIC (Akaike information criterion): Risk to inference

In [5]:
# Extract variables

risk_perception = df["mean_risk_perception_final_mean"].to_numpy()
inference = df["mean_inference_final_mean"].to_numpy()

# Logistic function

def logistic(x, L, k, x0):
    return L / (1 + np.exp(-k * (x - x0)))

# Fit logistic model

initial_parameters = [1, 5, np.median(risk_perception)]

parameters, _ = curve_fit(
    logistic,
    risk_perception,
    inference,
    p0=initial_parameters,
    maxfev=10000,
)

L, k, x0 = parameters

inference_pred_logistic = logistic(
    risk_perception,
    L,
    k,
    x0,
)

# Fit linear model for comparison

X_linear = sm.add_constant(risk_perception)

linear_model = sm.OLS(
    inference,
    X_linear,
).fit()

inference_pred_linear = linear_model.predict(X_linear)

# Calculate AIC

def calculate_aic(y, y_predicted, n_parameters):
    residuals = y - y_predicted
    sse = np.sum(residuals**2)
    n = len(y)

    return n * np.log(sse / n) + 2 * n_parameters


aic_linear = calculate_aic(
    inference,
    inference_pred_linear,
    2,
)

aic_logistic = calculate_aic(
    inference,
    inference_pred_logistic,
    3,
)

print("Logistic parameters:")
print(f"L = {L:.3f}")
print(f"k = {k:.3f}")
print(f"x0 = {x0:.3f}")

print("\nAIC comparison:")
print(f"Linear AIC:   {aic_linear:.2f}")
print(f"Logistic AIC: {aic_logistic:.2f}")
print(f"ΔAIC (linear - logistic): {aic_linear - aic_logistic:.2f}")

Logistic parameters:
L = 1.027
k = 11.447
x0 = 0.449

AIC comparison:
Linear AIC:   -138.23
Logistic AIC: -166.17
ΔAIC (linear - logistic): 27.94


### Spearman: Inference to global JS divergence

In [6]:
# Spearman correlation

r, p = spearmanr(
    df["mean_inference_final_mean"],
    df["mean_js_final_mean"],
)

print(
    f"Spearman(inference, global JS) = "
    f"{r:.3f}, p={p:.4g}"
)

Spearman(inference, global JS) = 0.821, p=8.759e-10


### Spearman: JS per feature spread

In [7]:
# Calculate JS divergence spread across features

df["js_spread"] = (
    df["mean_max_js_final_mean"]
    - df["mean_min_js_final_mean"]
)

r, p = spearmanr(
    df["mean_inference_final_mean"],
    df["js_spread"],
)

print(
    f"Spearman(inference, JS spread) = "
    f"{r:.3f}, p={p:.4g}"
)

Spearman(inference, JS spread) = 0.860, p=1.88e-11


## 2. CONTRAFACTUAL VARIANTS

### Wilcoxon: Inference variants

In [8]:
# Read model-variant results

variants = pd.read_csv(MODEL_VARIANTS)

column = "mean_js_final_mean"

variant_0 = variants[
    variants["model_variant"] == 0
]

variant_2 = variants[
    variants["model_variant"] == 2
]

# Align the two variants by social memory size and outgroup contact rate

pivot_0 = variant_0.pivot(
    index="social_memory_size",
    columns="p_outgroup",
    values=column,
)

pivot_2 = variant_2.pivot(
    index="social_memory_size",
    columns="p_outgroup",
    values=column,
)

aligned_0, aligned_2 = pivot_0.align(pivot_2)

differences = (
    aligned_2 - aligned_0
).stack().dropna()

# Wilcoxon signed-rank test

statistic, p = wilcoxon(differences)

print(f"W-statistic: {statistic}")
print(f"p-value: {p}")
print(f"Mean difference: {differences.mean()}")

W-statistic: 22.0
p-value: 1.5599653124809265e-08
Mean difference: 0.295017193705586
